### Data Fetching

In [ ]:
import psycopg2
import pandas as pd

In [ ]:
def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

In [ ]:
keywords = ["Projector"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df

In [ ]:
import pandas as pd

def rename_keys(d):
    if not isinstance(d, dict):
        return d
    
    mapping = {
        'ups_system': 'projector',
    }

    return {mapping.get(k, k): v for k, v in d.items()}

df['json_data'] = df['json_data'].apply(rename_keys)

valid_json = df['json_data'][df['json_data'].apply(lambda x: isinstance(x, dict))]
all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))


In [ ]:
#check the full content of json data for the first few rows
df

In [ ]:
import pandas as pd
import json

df_projector = df.copy(deep=True)

valid_mask = df_projector['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_projector[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename']
    }

    projector_data = row['json_data'].get('projector', {})

    for key, value in projector_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_projector = pd.DataFrame(rows)

df_projector = df_projector.drop(
    columns=["pm_order_no", "reference_document_no", "reference_document"],
    errors="ignore"
)

df_projector.rename(columns={
    'performed_by': 'technician_id',
    'verified_by': 'supervisor_id',
}, inplace=True)

df_projector.head()

In [ ]:
import json

df_projector["projector"] = df_projector["projector"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

df_projector["a"] = df_projector["projector"].apply(
    lambda x: x.get("a") if isinstance(x, dict) else None
)

df_projector["b"] = df_projector["projector"].apply(
    lambda x: x.get("b") if isinstance(x, dict) else None
)

df_projector.drop(columns=["projector"], inplace=True)

df_projector.head()

In [ ]:
def flatten_dict(d, parent_key='', sep='.'):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
            
    return items

a_flat = df_projector["a"].apply(
    lambda x: flatten_dict(x, parent_key="a") if isinstance(x, dict) else {}
)

b_flat = df_projector["b"].apply(
    lambda x: flatten_dict(x, parent_key="b") if isinstance(x, dict) else {}
)

df_b_flat = pd.json_normalize(b_flat)
df_a_flat = pd.json_normalize(a_flat)

df_final = pd.concat(
    [df_projector.drop(columns=["a", "b"]),
     df_a_flat,
     df_b_flat],
    axis=1
)

df_final.head()

In [ ]:
output_file = f"../../output/snc/projector.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='projector', index=False)

print(f"Saved excel to {output_file}")